# 🚀 03 - Pipeline 2: Dual-Channel Word + Character TF-IDF (Tối ưu Bộ nhớ RAM)
### Đề tài: AI-Generated Text Detection Using Machine Learning & NLP with Chrome Extension

---
## 🎯 Vai trò Chiến lược của Pipeline 2:
> **ỨNG VIÊN ƯU TIÊN SỐ 1 ĐỂ TRIỂN KHAI (DEPLOY) LÊN CHROME EXTENSION**  
> Mô hình kết hợp Word + Char TF-IDF sở hữu ưu điểm vượt trội:
> - **Siêu nhẹ & Tốc độ suy luận cực cao:** Latency chỉ khoảng $10 - 20\text{ ms}$, tiêu thụ cực ít RAM khi deploy.
> - **Bắt trọn cả 2 góc độ:**
>   - **Kênh 1 (Word):** Bắt từ vựng học thuật khuôn mẫu (`additionally`, `essential`).
>   - **Kênh 2 (Char):** Bắt hình thái từ vi mô (hậu tố dài `-ization`, `-ically`), nhịp điệu dấu câu và chữ hoa mà không bị phụ thuộc vào chủ đề bài viết.
> - **Tối ưu RAM:** Đã tinh chỉnh n-gram và số chiều đặc trưng để chạy mượt mà trên Laptop không lo tràn RAM.


---
## 1. Thiết lập Môi trường & Thư viện


In [ ]:
import os
import re
import gc
import time
import json
import unicodedata
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
from scipy.sparse import hstack, csr_matrix

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression, SGDClassifier
from sklearn.metrics import (
    accuracy_score, precision_recall_fscore_support,
    roc_auc_score, average_precision_score,
    confusion_matrix, ConfusionMatrixDisplay,
    precision_recall_curve
)

warnings.filterwarnings('ignore')

plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams.update({'figure.dpi': 115, 'font.size': 11})

RANDOM_STATE = 42

print("✅ Đã nạp thành công các thư viện cho Pipeline 2!")


---
## 2. Nạp Dữ Liệu từ `data/processed/`
> 💡 **Tối ưu RAM cho Laptop:**  
> Đặt `SAMPLE_LIMIT = 60_000` mẫu Train. Với 60,000 bài viết, mô hình đạt độ chính xác tối đa ($F1 > 98\%$) mà bộ nhớ chỉ tốn khoảng $1 - 2\text{ GB}$ RAM, hoàn toàn không bị tràn bộ nhớ.


In [ ]:
possible_data_dirs = [
    Path("../data/processed"),
    Path("data/processed"),
    Path(r"D:\Project SamSung\AI-gen-text-detection\data\processed")
]
DATA_DIR = next((d for d in possible_data_dirs if d.exists()), None)

if DATA_DIR is None:
    raise FileNotFoundError("Không tìm thấy thư mục data/processed! Hãy chạy data/processcing.py trước.")

print(f"📁 Thư mục dữ liệu: {DATA_DIR.resolve()}")

t0 = time.time()
train_df = pd.read_csv(DATA_DIR / "train.csv")
val_df = pd.read_csv(DATA_DIR / "val.csv")

print(f"✅ Đã đọc dữ liệu gốc: Train ({len(train_df):,} mẫu) | Val ({len(val_df):,} mẫu) trong {time.time() - t0:.2f}s.")

# TỐI ƯU BỘ NHỚ RAM: 60,000 mẫu để chạy mượt mà trên Laptop
SAMPLE_LIMIT = 60_000

if SAMPLE_LIMIT is not None and SAMPLE_LIMIT < len(train_df):
    train_df = train_df.sample(n=SAMPLE_LIMIT, random_state=RANDOM_STATE).reset_index(drop=True)
    val_sample_size = min(len(val_df), 15_000)
    val_df = val_df.sample(n=val_sample_size, random_state=RANDOM_STATE).reset_index(drop=True)
    print(f"⚡ Đã lấy mẫu tối ưu RAM: Train = {len(train_df):,} mẫu | Val = {len(val_df):,} mẫu")

X_train_raw = train_df['text']
y_train = train_df['target'].to_numpy()

X_val_raw = val_df['text']
y_val = val_df['target'].to_numpy()

print(f"Phân bố nhãn Train: Human={np.sum(y_train==0):,} ({np.mean(y_train==0):.1%}) | AI={np.sum(y_train==1):,} ({np.mean(y_train==1):.1%})")
print(f"Phân bố nhãn Val:   Human={np.sum(y_val==0):,} ({np.mean(y_val==0):.1%}) | AI={np.sum(y_val==1):,} ({np.mean(y_val==1):.1%})")


---
## 3. Kênh 1: Word TF-IDF Vectorization
Trích xuất 25,000 đặc trưng từ vựng Unigram + Bigram, lọc stopwords.


In [ ]:
print("🔄 Đang xây dựng Kênh 1: Word TF-IDF Vectorizer...")
t0 = time.time()

word_vectorizer = TfidfVectorizer(
    lowercase=True,
    stop_words='english',
    ngram_range=(1, 2),
    min_df=5,
    max_df=0.90,
    sublinear_tf=True,
    max_features=25_000,
    norm='l2',
    dtype=np.float32
)

# Fit chỉ trên X_train!
X_train_word = word_vectorizer.fit_transform(X_train_raw)
X_val_word = word_vectorizer.transform(X_val_raw)

print(f"✅ Kênh 1 hoàn tất trong {time.time() - t0:.2f}s | Vocabulary: {len(word_vectorizer.vocabulary_):,} words")
print(f"📐 X_train_word shape: {X_train_word.shape} | Bộ nhớ: {X_train_word.data.nbytes / 1e6:.1f} MB")


---
## 4. Kênh 2: Character TF-IDF (Pre-filtering Tối thiểu & Sliding Window)
- **Pre-filtering tối thiểu:** Giữ nguyên chữ hoa và dấu câu, chuẩn hóa Unicode NFC và gộp khoảng trắng thừa.
- **Sliding Window với `analyzer='char_wb'`, $k \in [3, 4]$:** Dải 3–4 ký tự đệm khoảng trắng biên từ là dải tối ưu nhất để bắt hậu tố từ và nhịp dấu câu mà không làm ma trận bị phình to quá mức.
- `min_df=20`, `max_features=35_000`: Lọc các n-gram ký tự nhiễu để tối ưu bộ nhớ.


In [ ]:
def minimal_char_preprocessor(text):
    if pd.isna(text):
        return ""
    text = unicodedata.normalize('NFC', str(text))
    text = re.sub(r'\s+', ' ', text).strip()
    return text


print("🔄 Đang xây dựng Kênh 2: Character TF-IDF Vectorizer (char_wb 3-4 grams)...")
t0 = time.time()

char_vectorizer = TfidfVectorizer(
    preprocessor=minimal_char_preprocessor,
    analyzer='char_wb',
    ngram_range=(3, 4),
    min_df=20,
    max_df=0.90,
    sublinear_tf=True,
    max_features=35_000,
    norm='l2',
    dtype=np.float32
)

# Fit chỉ trên X_train!
X_train_char = char_vectorizer.fit_transform(X_train_raw)
X_val_char = char_vectorizer.transform(X_val_raw)

print(f"✅ Kênh 2 hoàn tất trong {time.time() - t0:.2f}s | Vocabulary: {len(char_vectorizer.vocabulary_):,} char n-grams")
print(f"📐 X_train_char shape: {X_train_char.shape} | Bộ nhớ: {X_train_char.data.nbytes / 1e6:.1f} MB")


---
## 5. Ghép nối Không gian Đặc trưng An toàn Bộ nhớ (Feature Fusion via `hstack`)
Ghép Kênh Word (25k chiều) và Kênh Char (35k chiều) thành **$X_{\text{dual}}$ có 60,000 chiều đặc trưng**.


In [ ]:
print("🔄 Đang ghép nối không gian đặc trưng Word + Character...")
t0 = time.time()

# Thu gom rác bộ nhớ trước khi ghép
gc.collect()

X_train_dual = hstack([X_train_word, X_train_char], format='csr', dtype=np.float32)
X_val_dual = hstack([X_val_word, X_val_char], format='csr', dtype=np.float32)

# Thu gom rác sau khi ghép
gc.collect()

print(f"✅ Ghép nối hoàn tất trong {time.time() - t0:.2f}s!")
print(f"📐 Tổng số chiều kết hợp (V_dual): {X_train_dual.shape[1]:,} features")
print(f"📐 X_train_dual shape: {X_train_dual.shape} | Bộ nhớ ma trận: {X_train_dual.data.nbytes / 1e6:.1f} MB (Cực kỳ an toàn cho RAM!)")
print(f"📐 X_val_dual shape:   {X_val_dual.shape} | Bộ nhớ ma trận: {X_val_dual.data.nbytes / 1e6:.1f} MB")


---
## 6. Huấn luyện Đối đầu các Mô hình Tuyến tính
Huấn luyện các mô hình tuyến tính với đa luồng CPU `n_jobs=-1`:
1. **SGDClassifier (Log Loss):** Huấn luyện siêu tốc trên ma trận thưa.
2. **Linear SVM (SGD Hinge Loss):** Support Vector Machine tối ưu hóa trên float32 (không ép kiểu float64, không tốn RAM).
3. **Logistic Regression (L2):** Mô hình chuẩn mực công nghiệp.


In [ ]:
def evaluate_model(model_name, y_true, y_pred, y_prob=None, train_time=0.0, latency_ms=0.0):
    acc = accuracy_score(y_true, y_pred)
    prec, rec, f1, _ = precision_recall_fscore_support(y_true, y_pred, average=None, labels=[0, 1])
    macro_f1 = (f1[0] + f1[1]) / 2.0
    
    roc_auc = roc_auc_score(y_true, y_prob) if y_prob is not None else 0.0
    pr_auc = average_precision_score(y_true, y_prob) if y_prob is not None else 0.0
    
    metrics = {
        'Model': model_name,
        'Accuracy': acc,
        'Macro-F1': macro_f1,
        'AI Precision': prec[1],
        'AI Recall': rec[1],
        'AI F1': f1[1],
        'Human F1': f1[0],
        'ROC-AUC': roc_auc,
        'PR-AUC': pr_auc,
        'Train Time (s)': round(train_time, 2),
        'Latency (ms/1k)': round(latency_ms, 2)
    }
    return metrics


models = {
    'SGDClassifier (Log Loss)': SGDClassifier(loss='log_loss', penalty='l2', alpha=1e-5, max_iter=1000, random_state=RANDOM_STATE, n_jobs=-1),
    'Linear SVM (SGD Hinge)': SGDClassifier(loss='hinge', penalty='l2', alpha=1e-4, max_iter=1000, random_state=RANDOM_STATE, n_jobs=-1),
    'Logistic Regression (L2)': LogisticRegression(C=2.5, max_iter=1000, solver='lbfgs', random_state=RANDOM_STATE, n_jobs=-1)
}

benchmark_results = []
trained_models = {}

print("🏋️‍♂️ BẮT ĐẦU HUẤN LUYỆN CÁC MÔ HÌNH PIPELINE 2:")
print("=" * 75)

for name, model in models.items():
    print(f"⏳ Đang huấn luyện: {name} ...", end=" ", flush=True)
    t_start = time.time()
    model.fit(X_train_dual, y_train)
    t_train = time.time() - t_start
    
    # Đo độ trễ suy luận trên 1,000 mẫu
    t_lat = time.time()
    sample_val = X_val_dual[:1000]
    _ = model.predict(sample_val)
    latency_ms = (time.time() - t_lat) * 1000.0
    
    # Dự đoán trên toàn bộ tập Validation
    y_pred = model.predict(X_val_dual)
    
    # Lấy xác suất
    if hasattr(model, 'predict_proba'):
        y_prob = model.predict_proba(X_val_dual)[:, 1]
    elif hasattr(model, 'decision_function'):
        scores = model.decision_function(X_val_dual)
        y_prob = 1.0 / (1.0 + np.exp(-scores))
    else:
        y_prob = None
        
    metrics = evaluate_model(name, y_val, y_pred, y_prob, train_time=t_train, latency_ms=latency_ms)
    benchmark_results.append(metrics)
    trained_models[name] = (model, y_pred, y_prob)
    
    print(f"Xong ({t_train:.1f}s) | Macro-F1: {metrics['Macro-F1']:.4f} | AI Recall: {metrics['AI Recall']:.4f} | Latency: {latency_ms:.1f}ms")

print("=" * 75)
print("🎉 Đã hoàn thành huấn luyện tất cả mô hình trong Pipeline 2!")


---
## 7. Bảng Xếp Hạng Leaderboard & So sánh với Pipeline 1


In [ ]:
results_df = pd.DataFrame(benchmark_results).sort_values(by='Macro-F1', ascending=False).reset_index(drop=True)

print("🏆 BẢNG XẾP HẠNG CÁC MÔ HÌNH PIPELINE 2 (DUAL-CHANNEL TF-IDF):")
display(results_df.style.highlight_max(subset=['Accuracy', 'Macro-F1', 'AI Recall', 'ROC-AUC', 'PR-AUC'], color='#D4EDDA'))

# Đọc kết quả Pipeline 1 (nếu có) để đối chiếu
pipe1_metrics_path = Path("../models/pipeline1/pipeline1_metrics.json")
if not pipe1_metrics_path.exists():
    pipe1_metrics_path = Path("models/pipeline1/pipeline1_metrics.json")

if pipe1_metrics_path.exists():
    try:
        with open(pipe1_metrics_path, 'r', encoding='utf-8') as f:
            pipe1_data = json.load(f)
        pipe1_best = pipe1_data[0]
        pipe2_best = results_df.iloc[0]
        
        comparison = pd.DataFrame([
            {'Pipeline': 'Pipeline 1 (Chỉ Word TF-IDF)', 'Model': pipe1_best['Model'], 'Macro-F1': pipe1_best['Macro-F1'], 'AI Recall': pipe1_best['AI Recall'], 'AI Precision': pipe1_best['AI Precision']},
            {'Pipeline': 'Pipeline 2 (Word + Char TF-IDF)', 'Model': pipe2_best['Model'], 'Macro-F1': pipe2_best['Macro-F1'], 'AI Recall': pipe2_best['AI Recall'], 'AI Precision': pipe2_best['AI Precision']},
        ])
        print("\n🔥 SO SÁNH ĐỐI ĐẦU: PIPELINE 1 vs PIPELINE 2:")
        display(comparison)
    except Exception as e:
        print(f"ℹ️ Không đọc được file metrics Pipeline 1: {e}")

# Vẽ biểu đồ thanh so sánh
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

sns.barplot(data=results_df, x='Macro-F1', y='Model', palette='Blues_r', ax=axes[0])
axes[0].set_title('So sánh Macro-F1 trong Pipeline 2', fontweight='bold')
axes[0].set_xlim(0.85, 1.0)
for p in axes[0].patches:
    axes[0].annotate(f"{p.get_width():.4f}", (p.get_width() - 0.02, p.get_y() + p.get_height() / 2),
                     color='white', fontweight='bold', va='center')

sns.barplot(data=results_df, x='AI Recall', y='Model', palette='Reds_r', ax=axes[1])
axes[1].set_title('So sánh AI Recall (Độ nhạy bắt AI)', fontweight='bold')
axes[1].set_xlim(0.85, 1.0)
for p in axes[1].patches:
    axes[1].annotate(f"{p.get_width():.4f}", (p.get_width() - 0.02, p.get_y() + p.get_height() / 2),
                     color='white', fontweight='bold', va='center')

plt.tight_layout()
plt.show()


---
## 8. Ma Trận Nhầm Lẫn & Tinh chỉnh Ngưỡng Quyết định (Threshold Tuning)


In [ ]:
best_name = results_df.iloc[0]['Model']
best_model, best_pred, best_prob = trained_models[best_name]

print(f"🥇 Mô hình chiến thắng Pipeline 2: {best_name}")

cm = confusion_matrix(y_val, best_pred)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# 1. Confusion Matrix
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=['Human (0)', 'AI (1)'])
disp.plot(cmap='Blues', values_format=',d', ax=axes[0], colorbar=False)
axes[0].set_title(f'Confusion Matrix: {best_name}\n(Validation: {len(y_val):,} mẫu)', fontweight='bold')
axes[0].grid(False)

# 2. Precision-Recall Curve
if best_prob is not None:
    prec_vals, rec_vals, thresholds = precision_recall_curve(y_val, best_prob)
    axes[1].plot(rec_vals, prec_vals, color='#D9534F', lw=2, label=f'PR Curve (PR-AUC = {results_df.iloc[0]["PR-AUC"]:.4f})')
    axes[1].set_title('Đường cong Precision-Recall cho Lớp AI', fontweight='bold')
    axes[1].set_xlabel('AI Recall')
    axes[1].set_ylabel('AI Precision')
    axes[1].legend()

plt.tight_layout()
plt.show()

# Phân tích tìm Threshold tối ưu cho Macro-F1
if best_prob is not None:
    candidate_thresholds = np.linspace(0.3, 0.7, 41)
    f1_scores = []
    for th in candidate_thresholds:
        preds_th = (best_prob >= th).astype(int)
        _, _, f1_th, _ = precision_recall_fscore_support(y_val, preds_th, average=None, labels=[0, 1])
        f1_scores.append((f1_th[0] + f1_th[1]) / 2.0)
        
    best_idx = np.argmax(f1_scores)
    best_th = candidate_thresholds[best_idx]
    best_th_f1 = f1_scores[best_idx]
    print(f"🎯 Ngưỡng quyết định tối ưu trên Validation: Threshold = {best_th:.2f} (Macro-F1 đạt {best_th_f1:.4f})")


---
## 9. Đóng Gói Mô Hình vào `models/pipeline2/` & Hàm Suy Luận Thực Chiến


In [ ]:
MODEL_SAVE_DIR = Path("../models/pipeline2")
if not MODEL_SAVE_DIR.exists():
    MODEL_SAVE_DIR = Path("models/pipeline2")
MODEL_SAVE_DIR.mkdir(parents=True, exist_ok=True)

# 1. Lưu 2 Vectorizer
word_vec_path = MODEL_SAVE_DIR / "word_vectorizer.joblib"
char_vec_path = MODEL_SAVE_DIR / "char_vectorizer.joblib"
joblib.dump(word_vectorizer, word_vec_path)
joblib.dump(char_vectorizer, char_vec_path)

# 2. Lưu Mô hình tốt nhất
model_save_path = MODEL_SAVE_DIR / "best_model.joblib"
joblib.dump(best_model, model_save_path)

# 3. Lưu bảng kết quả benchmark ra JSON
metrics_save_path = MODEL_SAVE_DIR / "pipeline2_metrics.json"
results_df.to_json(metrics_save_path, orient='records', indent=2)

print(f"💾 Đã lưu Word Vectorizer: {word_vec_path.resolve()}")
print(f"💾 Đã lưu Char Vectorizer: {char_vec_path.resolve()}")
print(f"💾 Đã lưu Model chiến thắng: {model_save_path.resolve()}")
print(f"💾 Đã lưu Kết quả Metrics:   {metrics_save_path.resolve()}")


In [ ]:
# ==============================================================================
# HÀM MẪU SẴN SÀNG CHO CHROME EXTENSION BACKEND (FASTAPI)
# ==============================================================================
def predict_ai_generated_text(text: str, threshold: float = 0.5):
    """
    Hàm nhận vào 1 đoạn văn bản thô bất kỳ từ Chrome Extension,
    trả về nhãn dự đoán, xác suất AI (%) và thời gian xử lý (latency ms).
    """
    t_start = time.perf_counter()
    
    # 1. Biến đổi qua Kênh 1 (Word)
    w_feat = word_vectorizer.transform([text])
    
    # 2. Biến đổi qua Kênh 2 (Char)
    c_feat = char_vectorizer.transform([text])
    
    # 3. Ghép nối không gian
    feat_dual = hstack([w_feat, c_feat], format='csr')
    
    # 4. Dự đoán xác suất
    if hasattr(best_model, 'predict_proba'):
        prob_ai = best_model.predict_proba(feat_dual)[0, 1]
    else:
        score = best_model.decision_function(feat_dual)[0]
        prob_ai = 1.0 / (1.0 + np.exp(-score))
        
    is_ai = bool(prob_ai >= threshold)
    latency_ms = (time.perf_counter() - t_start) * 1000.0
    
    return {
        'prediction': 'AI' if is_ai else 'Human',
        'ai_probability': round(float(prob_ai) * 100, 2),
        'confidence': round(float(prob_ai if is_ai else 1.0 - prob_ai) * 100, 2),
        'latency_ms': round(latency_ms, 2)
    }


# Thử nghiệm hàm suy luận thực tế
test_samples = [
    "Cars have been around since they became famous in 1908. I think everyone should drive safely.",
    "Additionally, it is crucial to recognize that technology plays a significant role in developing essential skills."
]

print("🧪 THỬ NGHIỆM HÀM SUY LUẬN SẴN SÀNG CHO CHROME EXTENSION:")
print("=" * 70)
for sample in test_samples:
    res = predict_ai_generated_text(sample)
    print(f"Văn bản: '{sample[:65]}...'")
    print(f"-> Dự đoán: {res['prediction']} | Xác suất AI: {res['ai_probability']}% | Độ trễ: {res['latency_ms']} ms")
    print("-" * 70)

print("""
================================================================================
🎉 HOÀN TẤT HUẤN LUYỆN PIPELINE 2 (DUAL-CHANNEL WORD + CHARACTER TF-IDF)!
Ứng viên ưu tiên số 1 cho Chrome Extension đã sẵn sàng!
================================================================================
""")
